# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emaanzehra/FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

My rule scores pages that are stale (not updated recently) and still visible (getting impressions). The reasoning is: a page that still attracts search traffic but has not been updated in a long time is a high-priority refresh candidate. The reason code is stale_visible_page. The action is refresh_review.

In [1]:
import os
os.environ["HF_TOKEN"] = __import__("google.colab", fromlist=["userdata"]).userdata.get("HF_TOKEN")
import duckdb, pandas as pd
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")
REL = "hf://datasets/FlyRank/internship-warehouse"

df = con.sql(f"""
SELECT content_hash_id,
       AVG(gsc_impressions) as avg_impressions,
       AVG(gsc_avg_position) as avg_position,
       AVG(gsc_clicks) as avg_clicks,
       COUNT(DISTINCT report_date) as days_with_data,
       MAX(gsc_impressions) as max_impressions
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE (gsc_impressions IS NOT NULL) IS TRUE
GROUP BY content_hash_id
""").df()

# Signal 1: staleness proxy (pages with fewer active days = more stale)
print("Signal 1: Days with data distribution (staleness proxy)")
print(df["days_with_data"].describe())
bins = [0,5,10,15,20,25,31]
df["staleness_bucket"] = pd.cut(df["days_with_data"], bins=bins)
print(df.groupby("staleness_bucket", observed=True)["avg_impressions"].agg(["mean","count"]))
print("VERDICT: CONFIRMED — pages with fewer active days tend to have lower impressions")

# Signal 2: position vs clicks
print("\nSignal 2: Position tier vs avg clicks")
df["position_tier"] = pd.cut(df["avg_position"], bins=[0,3,10,20,50,200])
print(df.groupby("position_tier", observed=True)["avg_clicks"].agg(["mean","count"]))
print("VERDICT: CONFIRMED — higher position (lower number) means more clicks")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1: Days with data distribution (staleness proxy)
count    331437.000000
mean         29.693058
std           4.738734
min           1.000000
25%          31.000000
50%          31.000000
75%          31.000000
max          31.000000
Name: days_with_data, dtype: float64
                       mean   count
staleness_bucket                   
(0, 5]             0.040115    5397
(5, 10]            1.025654    1744
(10, 15]           8.137239    4372
(15, 20]          21.765989    5692
(20, 25]           8.239000    2043
(25, 31]          28.694414  312189
VERDICT: CONFIRMED — pages with fewer active days tend to have lower impressions

Signal 2: Position tier vs avg clicks
                   mean  count
position_tier                 
(0, 3]         0.324730  16144
(3, 10]        0.189636  81988
(10, 20]       0.108121  32203
(20, 50]       0.077029  33288
(50, 200]      0.002332  11660
VERDICT: CONFIRMED — higher position (lower number) means more clicks


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import os, pathlib
pathlib.Path("work/outputs").mkdir(parents=True, exist_ok=True)

df["stale"] = (df["days_with_data"] <= 15).astype(int)
df["visible"] = (df["avg_impressions"] >= 10).astype(int)
df["baseline_score"] = df["stale"] * df["visible"] * df["avg_impressions"]
df["reason_code"] = ""
df.loc[(df["stale"]==1) & (df["visible"]==1), "reason_code"] = "stale_visible_page"
df["action"] = ""
df.loc[df["reason_code"]=="stale_visible_page", "action"] = "refresh_review"
queue = df.sort_values("baseline_score", ascending=False)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Queue written: {len(queue)} rows")
print(f"Pages flagged for refresh_review: {(queue['action']=='refresh_review').sum()}")
print(queue[["content_hash_id","baseline_score","reason_code","action"]].head(10))

Queue written: 331437 rows
Pages flagged for refresh_review: 641
                 content_hash_id  baseline_score         reason_code  \
326856  content_0c11901fc5d75ff9      861.846154  stale_visible_page   
160208  content_e91de019c0c07462      620.076923  stale_visible_page   
161760  content_6af00a7fbd3def79      539.769231  stale_visible_page   
327190  content_35868bbe5afbce53      514.307692  stale_visible_page   
326158  content_0fe39356beb1411a      504.769231  stale_visible_page   
161213  content_779394c90b8743ae      483.384615  stale_visible_page   
160400  content_ad53d798d53b98a6      426.384615  stale_visible_page   
161416  content_ea6db3ecfe821513      374.461538  stale_visible_page   
326945  content_5adb95135ab735e0      355.307692  stale_visible_page   
160383  content_b09ffe56d1bbf3af      323.000000  stale_visible_page   

                action  
326856  refresh_review  
160208  refresh_review  
161760  refresh_review  
327190  refresh_review  
326158  refresh_r

## 3. Top-20 review

The top 10 pages all carry the stale_visible_page reason code, meaning they had fewer than 15 active days in March 2026 but still received meaningful impressions. Each is flagged for refresh_review. What would make any of these wrong: if the page was recently republished without a metadata date change, the staleness signal would be misleading. If impressions came from a single spike rather than sustained traffic, visibility would be inflated. If the page is intentionally evergreen content that needs no update, the recommendation wastes reviewer time.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

Weak picks: pages with very low impression counts but technically meeting the stale threshold could appear in the queue unfairly. The threshold of 10 average impressions is low and may include noisy pages. No product flags or future-window columns were used. The score uses only days_with_data and avg_impressions, both observable before any decision is made. The leaky_ctr_ratio column built in ML-04 was not used here.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.